# 1. Research Question

Are traders executing larger transactions more profitable? Is variability in transaction size associated with profitability?

The primary outcome is **profit per unit of volume (`trader_ppv`)**, with **total profit/loss (`trader_pnl`)** as a secondary outcome. We analyze traders with at least 100 transactions, giving each trader equal weight. We also compare traders within total-volume groups and examine whether PPV's mathematical construction helps explain associations. This observational analysis does not identify causal effects.

# 2. Data Loading

Run this notebook from the folder containing `data_no_imputation.parquet`. It loads its own data and never writes to the source Parquet file. Dependencies: pandas, NumPy, Matplotlib, SciPy, IPython, and a Parquet engine such as PyArrow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (11, 6), "font.size": 11})
pd.set_option("display.max_columns", 20)

df = pd.read_parquet("data_no_imputation.parquet")
print(f"Dataset shape: {df.shape[0]:,} rows ? {df.shape[1]:,} columns")
required_columns = [
    "mean_tx_value", "std_tx_value", "trader_ppv", "trader_pnl",
    "transaction_count", "trader_volume",
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
print("All required columns exist.")
display(df[required_columns].dtypes.rename("dtype").to_frame())

# 3. Data Preparation

Keep traders with `transaction_count >= 100`, then exclude missing or infinite values in the six relevant columns. Preserve every extreme but finite observation. Negative transaction sizes, standard deviations, or trading volumes cause an explicit validation error rather than silent deletion. Zero values remain in the sample.

If trader identifiers are available, check that they are unique before treating rows as separate traders. Missing or repeated identifiers require investigation.

In [ ]:
if "trader" in df.columns:
    identifiers = df["trader"].astype("string").str.strip()
    missing_ids = identifiers.isna() | identifiers.eq("").fillna(False)
    duplicate_ids = identifiers.loc[~missing_ids].duplicated(keep=False)
    print(f"Missing/blank trader IDs: {missing_ids.sum():,}; rows with repeated IDs: {duplicate_ids.sum():,}")
    if missing_ids.any() or duplicate_ids.any():
        raise ValueError("Resolve missing or repeated trader identifiers before trader-level analysis.")
    print("Trader identifiers are unique.")
else:
    print("No trader column: one row per trader is an unverified assumption.")

analysis_df = df[required_columns].copy()
for column in required_columns:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="raise")
analysis_df = analysis_df.loc[analysis_df["transaction_count"] >= 100].copy()
threshold_count = len(analysis_df)
finite_rows = np.isfinite(analysis_df.to_numpy(dtype=float, na_value=np.nan)).all(axis=1)
analysis_df = analysis_df.loc[finite_rows].copy()
print(f"Traders meeting the transaction threshold: {threshold_count:,}")
print(f"Removed for missing/infinite relevant values: {threshold_count - len(analysis_df):,}")
print(f"Traders remaining: {len(analysis_df):,}")
if len(analysis_df) < 3:
    raise ValueError("At least three valid traders are required.")
for column in ["mean_tx_value", "std_tx_value", "trader_volume"]:
    if (analysis_df[column] < 0).any():
        raise ValueError(f"Negative values found in {column}; inspect source data.")
if analysis_df["mean_tx_value"].nunique() < 2:
    raise ValueError("Average transaction size has no variation.")
display(analysis_df.describe(percentiles=[0.01, 0.5, 0.9, 0.99]))

In [ ]:
def quantile_groups(values, requested=10):
    # Preserve ties rather than arbitrarily allocating identical values to different bins.
    if values.nunique() < 2:
        return pd.Series(pd.Categorical(["All traders"] * len(values)), index=values.index)
    return pd.qcut(values, q=requested, duplicates="drop")

analysis_df["size_group"] = quantile_groups(analysis_df["mean_tx_value"])
summary_table = (
    analysis_df.groupby("size_group", observed=True)
    .agg(
        n_traders=("trader_ppv", "size"),
        median_mean_tx_value=("mean_tx_value", "median"),
        min_mean_tx_value=("mean_tx_value", "min"),
        max_mean_tx_value=("mean_tx_value", "max"),
        mean_trader_ppv=("trader_ppv", "mean"),
        median_trader_ppv=("trader_ppv", "median"),
        mean_trader_pnl=("trader_pnl", "mean"),
        median_trader_pnl=("trader_pnl", "median"),
    ).reset_index()
)
summary_table.insert(0, "group", [f"Q{i+1}" for i in range(len(summary_table))])
assert summary_table["n_traders"].sum() == len(analysis_df)
print(f"Requested 10 size quantiles; obtained {len(summary_table)} nonempty groups.")
print("Tied quantile boundaries can reduce group count; ties can also produce unequal group sizes.")
display(summary_table.round(6))

# 4. Visualizations

Each chart is a separate figure. To assess readability, report each positive X range and use a log axis if it spans at least two orders of magnitude. Where zeros coexist with that broad range, use a symmetric-log axis with a linear region near zero. Axis transformations retain all observations. Profit axes stay linear so losses, gains, and finite extremes remain visible.

Charts 2 and 3 plot group profitability against the **median of traders' average transaction sizes** in each group. Labels show group identifiers and trader counts; exact ranges appear in the summary table.

In [ ]:
axis_settings = {}
for column in ["mean_tx_value", "std_tx_value"]:
    values = analysis_df[column].to_numpy(dtype=float)
    positive = values[values > 0]
    ratio = positive.max() / positive.min() if positive.size else 1.0
    zeros = int((values == 0).sum())
    scale = ("symlog" if zeros else "log") if ratio >= 100 else "linear"
    axis_settings[column] = (scale, float(positive.min()) if positive.size else 1.0)
    print(f"{column}: range {values.min():.6g} to {values.max():.6g}; positive max/min {ratio:,.1f}; zeros {zeros:,}; X scale {scale}")

def format_axis(ax, column, label):
    scale, threshold = axis_settings[column]
    if scale == "symlog":
        ax.set_xscale(scale, linthresh=threshold)
    else:
        ax.set_xscale(scale)
    ax.set_xlabel(f"{label} (dataset monetary units; {scale} scale)")
    ax.set_ylabel("Profit per unit of trading volume (trader_ppv)")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.grid(alpha=0.2)

fig, ax = plt.subplots()
ax.scatter(analysis_df["mean_tx_value"], analysis_df["trader_ppv"], alpha=0.15, s=12, edgecolors="none")
ax.set_title("Average transaction size vs. profitability | 100+ transactions")
format_axis(ax, "mean_tx_value", "Average transaction size")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(summary_table["median_mean_tx_value"], summary_table["mean_trader_ppv"], marker="o")
for i, row in summary_table.iterrows():
    ax.annotate(
        f"{row['group']}\nn={row['n_traders']:,}",
        (row["median_mean_tx_value"], row["mean_trader_ppv"]),
        xytext=(0, 12 if i % 2 == 0 else -32), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_title("Mean profitability by average transaction-size quantile group")
format_axis(ax, "mean_tx_value", "Median average transaction size within group")
ax.set_ylabel("Mean trader_ppv (equal trader weights)")
ax.margins(y=0.3)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(summary_table["median_mean_tx_value"], summary_table["median_trader_ppv"], marker="o")
for i, row in summary_table.iterrows():
    ax.annotate(
        f"{row['group']}\nn={row['n_traders']:,}",
        (row["median_mean_tx_value"], row["median_trader_ppv"]),
        xytext=(0, 12 if i % 2 == 0 else -32), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_title("Median profitability by average transaction-size quantile group")
format_axis(ax, "mean_tx_value", "Median average transaction size within group")
ax.set_ylabel("Median trader_ppv (equal trader weights)")
ax.margins(y=0.3)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(analysis_df["std_tx_value"], analysis_df["trader_ppv"], alpha=0.15, s=12, edgecolors="none")
ax.set_title("Transaction-size variability vs. profitability | 100+ transactions")
format_axis(ax, "std_tx_value", "Standard deviation of transaction size")
fig.tight_layout()
plt.show()

# 5. Statistical Analysis

Pearson measures linear association; Spearman measures rank association. Correlations use the original, untransformed individual-trader values. Report both PPV and total P&L for average size and size variability. P-values are nominal, unadjusted for multiple comparisons, and assume independent observations. Unique wallet IDs do not guarantee independent owners; small p-values need not imply economically large effects.

In [ ]:
def correlation_record(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    record = {"n_traders": len(x)}
    for name, function in [("pearson", stats.pearsonr), ("spearman", stats.spearmanr)]:
        if len(x) < 3 or np.unique(x).size < 2 or np.unique(y).size < 2:
            coefficient, p_value = np.nan, np.nan
        else:
            coefficient, p_value = function(x, y)
        record[f"{name}_r"] = coefficient
        record[f"{name}_p"] = p_value
    return record

correlation_table = pd.DataFrame([
    {"predictor": predictor, "outcome": outcome,
     **correlation_record(analysis_df[predictor], analysis_df[outcome])}
    for predictor in ["mean_tx_value", "std_tx_value"]
    for outcome in ["trader_ppv", "trader_pnl"]
])
display(correlation_table.style.format({
    "pearson_r": "{:.6f}", "spearman_r": "{:.6f}",
    "pearson_p": "{:.6g}", "spearman_p": "{:.6g}", "n_traders": "{:,.0f}",
}))

### Comparing traders with similar total trading volume

Stratify traders into 10 total-volume quantiles and compute average-size correlations within each stratum, for both PPV and P&L. Compare direction and magnitude with the pooled correlations. This is a descriptive check of persistence, not a causal adjustment. Quantile ranges can remain broad, especially in the highest-volume group; inspect their minimum and maximum values. Conditional associations can differ because volume, transaction count, and mean size are mechanically connected.

In [ ]:
analysis_df["volume_group"] = quantile_groups(analysis_df["trader_volume"])
volume_rows = []
for i, (group, sample) in enumerate(analysis_df.groupby("volume_group", observed=True), 1):
    for outcome in ["trader_ppv", "trader_pnl"]:
        volume_rows.append({
            "volume_group": f"V{i}", "outcome": outcome,
            "min_volume": sample["trader_volume"].min(),
            "median_volume": sample["trader_volume"].median(),
            "max_volume": sample["trader_volume"].max(),
            **correlation_record(sample["mean_tx_value"], sample[outcome]),
        })
volume_correlations = pd.DataFrame(volume_rows)
print(f"Nonempty total-volume strata: {volume_correlations['volume_group'].nunique()}")
display(volume_correlations.style.format({
    "min_volume": "{:,.2f}", "median_volume": "{:,.2f}", "max_volume": "{:,.2f}",
    "pearson_r": "{:.5f}", "spearman_r": "{:.5f}",
    "pearson_p": "{:.4g}", "spearman_p": "{:.4g}", "n_traders": "{:,.0f}",
}))

### Absolute versus relative size consistency

Standard deviation measures absolute variability and tends to grow with typical transaction size. As a supplemental check, use the coefficient of variation, `std_tx_value / mean_tx_value`, to describe relative variability. Compute it only for positive means and finite ratios; excluded ratios do not remove traders from the main analysis. Lower CV indicates more consistent sizes relative to the average. Neither SD nor CV alone establishes a strategy's quality.

In [ ]:
with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    cv = analysis_df["std_tx_value"] / analysis_df["mean_tx_value"]
valid_cv = (analysis_df["mean_tx_value"] > 0) & np.isfinite(cv)
cv_df = analysis_df.loc[valid_cv, ["trader_ppv", "trader_pnl"]].copy()
cv_df["size_cv"] = cv.loc[valid_cv]
print(f"Relative-variability sample: {len(cv_df):,}; undefined/nonfinite ratios: {(~valid_cv).sum():,}")
cv_correlations = pd.DataFrame([
    {"predictor": "std_tx_value / mean_tx_value", "outcome": outcome,
     **correlation_record(cv_df["size_cv"], cv_df[outcome])}
    for outcome in ["trader_ppv", "trader_pnl"]
])
display(cv_correlations)
if not cv_df.empty:
    cv_df["cv_group"] = quantile_groups(cv_df["size_cv"], requested=5)
    cv_summary = cv_df.groupby("cv_group", observed=True).agg(
        n_traders=("trader_ppv", "size"), median_cv=("size_cv", "median"),
        mean_trader_ppv=("trader_ppv", "mean"), median_trader_ppv=("trader_ppv", "median"),
        mean_trader_pnl=("trader_pnl", "mean"), median_trader_pnl=("trader_pnl", "median"),
    )
    display(cv_summary.round(6))

### Mathematical construction of profitability

Check, rather than assume, whether `trader_ppv ? trader_pnl / trader_volume` and `mean_tx_value ? trader_volume / transaction_count`. Report agreement within a numerical tolerance. These checks describe the available values and do not establish the upstream feature definitions.

If these identities hold, PPV and average size share volume in opposite positions: at fixed P&L and transaction count, increasing volume increases mean size and decreases PPV. This can contribute to an association even without a behavioral explanation; it does not dictate the observed correlation because P&L and count also vary. Within a narrow volume stratum, larger average size also implies fewer transactions when the second identity holds. P&L avoids the volume denominator but still reflects scale and exposure.

In [ ]:
identity_rows = []
for label, numerator, denominator, observed in [
    ("PPV vs P&L / volume", "trader_pnl", "trader_volume", "trader_ppv"),
    ("Mean size vs volume / count", "trader_volume", "transaction_count", "mean_tx_value"),
]:
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        reconstructed = analysis_df[numerator] / analysis_df[denominator]
    eligible = (analysis_df[denominator] > 0) & np.isfinite(reconstructed)
    actual = analysis_df.loc[eligible, observed].to_numpy(dtype=float)
    predicted = reconstructed.loc[eligible].to_numpy(dtype=float)
    close = np.isclose(actual, predicted, rtol=1e-5, atol=1e-8)
    identity_rows.append({
        "check": label, "n_checked": len(actual), "n_excluded": int((~eligible).sum()),
        "fraction_matching": float(close.mean()) if len(close) else np.nan,
        "median_absolute_difference": float(np.median(np.abs(actual - predicted))) if len(actual) else np.nan,
    })
identity_checks = pd.DataFrame(identity_rows)
print("Matching tolerance: rtol=1e-5, atol=1e-8")
display(identity_checks)

# 6. Key Findings

The findings below are computed from the current run. Judge effect sizes alongside p-values. Differences between the mean and median reveal sensitivity to asymmetric outcomes; all finite extremes remain included. Within-volume correlations are descriptive and do not remove all confounding by experience, strategy, risk, or exposure.

In [ ]:
findings = [f"**Sample:** {len(analysis_df):,} traders with at least 100 transactions; {len(summary_table)} transaction-size groups."]
for _, row in correlation_table.iterrows():
    findings.append(
        f"**{row['predictor']} vs {row['outcome']}:** Pearson r={row['pearson_r']:.4f} "
        f"(p={row['pearson_p']:.3g}); Spearman rho={row['spearman_r']:.4f} "
        f"(p={row['spearman_p']:.3g}); n={row['n_traders']:,}."
    )
first, last = summary_table.iloc[0], summary_table.iloc[-1]
findings.append(
    f"**Smallest to largest size group:** mean PPV {first['mean_trader_ppv']:.6f} ? {last['mean_trader_ppv']:.6f}; "
    f"median PPV {first['median_trader_ppv']:.6f} ? {last['median_trader_ppv']:.6f}."
)
for outcome in ["trader_ppv", "trader_pnl"]:
    within = volume_correlations.loc[volume_correlations['outcome'] == outcome, 'spearman_r'].dropna()
    pooled = correlation_table.loc[(correlation_table['predictor'] == 'mean_tx_value') & (correlation_table['outcome'] == outcome), 'spearman_r'].iloc[0]
    same_direction = int(((within * pooled) > 0).sum()) if np.isfinite(pooled) else 0
    findings.append(
        f"**Within-volume size association with {outcome}:** Spearman rho ranges from "
        f"{within.min():.4f} to {within.max():.4f}; {same_direction}/{len(within)} defined strata "
        "have the same sign as the pooled coefficient. Compare magnitudes in the table; matching signs alone do not establish robustness."
    )
cv_row = cv_correlations.loc[cv_correlations['outcome'] == 'trader_ppv'].iloc[0]
findings.append(f"**Relative size variability:** CV vs PPV Spearman rho={cv_row['spearman_r']:.4f} (p={cv_row['spearman_p']:.3g}); compare this with absolute-SD results and the CV-group table.")
for _, row in identity_checks.iterrows():
    findings.append(f"**{row['check']}:** {row['fraction_matching']:.1%} of {row['n_checked']:,} eligible rows match within the stated tolerance.")
mean_identity_match = identity_checks.loc[identity_checks["check"] == "Mean size vs volume / count", "fraction_matching"].iloc[0]
if mean_identity_match < 0.99:
    findings.append("**Feature-definition caution:** average transaction size does not consistently equal total volume divided by transaction count in this dataset. Do not assume that identity or its implied fixed-volume size/count tradeoff; the upstream definitions need clarification.")
findings.append("**Interpretation:** denominator coupling, scale, and residual differences between traders limit interpretation. These results do not show that changing transaction size or consistency causes profitability to change.")
display(Markdown("\n".join("- " + finding for finding in findings)))